In [ ]:

import os
import urllib.parse
import prettytable
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))

SERVER   = os.getenv("DB_SERVER")
DATABASE = os.getenv("DB_NAME", "OlistDB")
USER     = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
DRIVER   = os.getenv("DB_DRIVER", "ODBC Driver 17 for SQL Server")


missing = [k for k, v in {"SERVER": SERVER, "USER": USER, "PASSWORD": PASSWORD}.items() if not v]
if missing:
    raise EnvironmentError(f"❌ Variáveis ausentes no .env: {', '.join(missing)}")

print(f"📦 Servidor : {SERVER}")
print(f"📦 Banco    : {DATABASE}")
print(f"📦 Usuário  : {USER}")


_available_styles = [k for k in prettytable.__dict__ if k.isupper() and isinstance(prettytable.__dict__[k], int)]
_preferred = ["DEFAULT", "SINGLE_BORDER", "MARKDOWN", "PLAIN_COLUMNS"]
PRETTY_STYLE = next((s for s in _preferred if s in _available_styles), _available_styles[0] if _available_styles else "DEFAULT")

print(f"🎨 PrettyTable style: {PRETTY_STYLE}  (disponíveis: {_available_styles})")

odbc_str = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USER};"
    f"PWD={PASSWORD};"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
    f"MARS_Connection=yes;"
)

connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(odbc_str)}"


%reload_ext sql
%config SqlMagic.feedback    = True         # Exibe quantidade de linhas retornadas
%config SqlMagic.autopandas  = False        # Retorna ResultSet (mais leve)
%config SqlMagic.displaycon  = False        # Oculta string de conexão no output
%config SqlMagic.style       = PRETTY_STYLE # Fix: usa estilo detectado dinamicamente


%sql {connection_url}

print("\n✅ Conexão estabelecida com OlistDB!")
print("   Use %%sql em uma célula separada para executar queries.")

📦 Servidor : LUCAS
📦 Banco    : OlistDB
📦 Usuário  : sa
🎨 PrettyTable style: DEFAULT  (disponíveis: ['ALL', 'DEFAULT', 'DOUBLE_BORDER', 'FRAME', 'HEADER', 'MARKDOWN', 'MSWORD_FRIENDLY', 'NONE', 'ORGMODE', 'PLAIN_COLUMNS', 'RANDOM', 'SINGLE_BORDER'])

✅ Conexão estabelecida com OlistDB!
   Use %%sql em uma célula separada para executar queries.


In [5]:
%%sql
SELECT
    t.name        AS tabela,
    c.name        AS coluna,
    tp.name       AS tipo,
    c.max_length  AS tamanho,
    c.is_nullable AS permite_nulo
FROM sys.tables t
JOIN sys.columns c  ON c.object_id = t.object_id
JOIN sys.types  tp  ON tp.user_type_id = c.user_type_id
ORDER BY t.name, c.column_id;

Done.


tabela,coluna,tipo,tamanho,permite_nulo
customers,customer_id,varchar,32,False
customers,customer_unique_id,varchar,32,False
customers,customer_zip_code_prefix,int,4,False
customers,customer_city,varchar,100,True
customers,customer_state,char,2,True
geolocation,geolocation_zip_code_prefix,int,4,False
geolocation,geolocation_lat,float,8,True
geolocation,geolocation_lng,float,8,True
geolocation,geolocation_city,varchar,100,True
geolocation,geolocation_state,char,2,True


In [6]:
%%sql
SELECT 
    t.name  AS tabela,
    c.name  AS coluna,
    tp.name AS tipo,
    c.is_nullable AS permite_nulo
FROM sys.tables t
JOIN sys.columns c  ON c.object_id = t.object_id
JOIN sys.types  tp  ON tp.user_type_id = c.user_type_id
WHERE t.name = 'products' 
  AND c.name IN ('product_name_lenght', 'product_description_lenght', 'product_photos_qty')
ORDER BY c.column_id;

Done.


tabela,coluna,tipo,permite_nulo
products,product_name_lenght,smallint,True
products,product_description_lenght,smallint,True
products,product_photos_qty,tinyint,True


In [7]:
%%sql
SELECT 
    t.name  AS tabela,
    c.name  AS coluna,
    tp.name AS tipo,
    c.precision AS precisao,
    c.scale AS escala
FROM sys.tables t
JOIN sys.columns c  ON c.object_id = t.object_id
JOIN sys.types  tp  ON tp.user_type_id = c.user_type_id
WHERE (t.name = 'order_items' AND c.name IN ('price', 'freight_value'))
   OR (t.name = 'order_payments' AND c.name = 'payment_value')
ORDER BY t.name, c.column_id;

Done.


tabela,coluna,tipo,precisao,escala
order_items,price,decimal,10,2
order_items,freight_value,decimal,10,2
order_payments,payment_value,decimal,10,2


In [8]:
%%sql
SELECT 
    t.name  AS tabela,
    c.name  AS coluna,
    tp.name AS tipo,
    c.max_length AS tamanho_bytes
FROM sys.tables t
JOIN sys.columns c  ON c.object_id = t.object_id
JOIN sys.types  tp  ON tp.user_type_id = c.user_type_id
WHERE t.name = 'order_reviews' 
  AND c.name = 'review_comment_title';

Done.


tabela,coluna,tipo,tamanho_bytes
order_reviews,review_comment_title,varchar,200
